In [1]:
"""
modified from the script found here:
    https://github.com/bmurauer/authbench/blob/main/scripts/unify_c50.py

dataset access:
    https://archive.ics.uci.edu/ml/datasets/Reuter_50_50
"""
from sklearn.model_selection import train_test_split
import logging
from tqdm import tqdm
import os
import argparse
import random
import numpy as np
from glob import glob
from typing import List, Dict
import re

from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktParameters

def process_c50(pth, seed=0):
    processed_dir = os.path.join(pth, "processed")
    if not os.path.isdir(processed_dir):
        os.makedirs(processed_dir)
    raw_dir = pth

    train = os.path.join(raw_dir, "C50train")
    test = os.path.join(raw_dir, "C50test")

    def read(subdir: str, author_ids: dict) -> List[Dict]:
        posts = {}
        authors = sorted(os.listdir(subdir))
        for j, author in enumerate(authors):
            author_dir = os.path.join(subdir, author)
            files = sorted(glob(author_dir + "/*.txt"))
            for f in files:
                with open(f) as i_f:
                    text = i_f.read()
                    posts.setdefault(author_ids.setdefault(author, j), []).append(text)
        return posts, author_ids

    logging.info('getting train and test sets')
    auth_to_id = {}  # make sure author id's are consistent across train and test set
    train_and_eval_dict, auth_to_id = read(train, auth_to_id)
    print(auth_to_id)
    test_dict, auth_to_id = read(test, auth_to_id)

    # make a dict of all data for stat tracking
    all_data = {}
    for data in [train_and_eval_dict, test_dict]:
        for k, v in data.items():
            for t in v:
                all_data.setdefault(k, []).append(t)

    # we need to split the train into a training and evaluation set
    train_and_eval_data = []
    for auth in train_and_eval_dict.keys():
        for text in train_and_eval_dict[auth]:
            train_and_eval_data.append([auth, text])

    logging.info(f'splitting the training data into train/eval sets')
    # now split into stratified train(60%)/val(20%)/test(20%) splits
    train_set, eval_set = train_test_split(train_and_eval_data, test_size=0.2, shuffle=True, random_state=seed,
                                                    stratify=[lbl for lbl, _ in train_and_eval_data])

    # now transform back to dicts
    train_dict = {}
    for auth, text in train_set:
        train_dict.setdefault(auth, []).append(text)

    val_dict = {}
    for auth, text in eval_set:
        val_dict.setdefault(auth, []).append(text)

    return train_dict, val_dict, test_dict


dataset_path = "./ccat50/"
seed = 0
output_path = "./ccat50/processed"

class Namespace:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

args = Namespace(dataset_path=dataset_path, seed=seed)

random.seed(args.seed)
np.random.seed(args.seed)

train_data, val_data, test_data = process_c50(args.dataset_path, args.seed)

{'AaronPressman': 0, 'AlanCrosby': 1, 'AlexanderSmith': 2, 'BenjaminKangLim': 3, 'BernardHickey': 4, 'BradDorfman': 5, 'DarrenSchuettler': 6, 'DavidLawder': 7, 'EdnaFernandes': 8, 'EricAuchard': 9, 'FumikoFujisaki': 10, 'GrahamEarnshaw': 11, 'HeatherScoffield': 12, 'JanLopatka': 13, 'JaneMacartney': 14, 'JimGilchrist': 15, 'JoWinterbottom': 16, 'JoeOrtiz': 17, 'JohnMastrini': 18, 'JonathanBirt': 19, 'KarlPenhaul': 20, 'KeithWeir': 21, 'KevinDrawbaugh': 22, 'KevinMorrison': 23, 'KirstinRidley': 24, 'KouroshKarimkhany': 25, 'LydiaZajc': 26, "LynneO'Donnell": 27, 'LynnleyBrowning': 28, 'MarcelMichelson': 29, 'MarkBendeich': 30, 'MartinWolk': 31, 'MatthewBunce': 32, 'MichaelConnor': 33, 'MureDickie': 34, 'NickLouth': 35, 'PatriciaCommins': 36, 'PeterHumphrey': 37, 'PierreTran': 38, 'RobinSidel': 39, 'RogerFillion': 40, 'SamuelPerry': 41, 'SarahDavison': 42, 'ScottHillis': 43, 'SimonCowell': 44, 'TanEeLyn': 45, 'TheresePoletti': 46, 'TimFarrand': 47, 'ToddNissen': 48, 'WilliamKazer': 49}


In [2]:
import re
import pickle

def split_sentence(sentence):
    # Create a Punkt tokenizer with custom parameters
    punkt_param = PunktParameters()
    abbreviation = ['corp', 'co', "u.s", "inc", "cos", "u.k", "st"]
    punkt_param.abbrev_types = set(abbreviation)
    tokenizer = PunktSentenceTokenizer(punkt_param)
    
    # Tokenize the sentence using the Punkt tokenizer
    sentences = tokenizer.tokenize(sentence)
    
    if len(sentences) > 1:
        first_part = sentences[0]
        second_part = ' '.join(sentences[1:])
        return first_part, second_part
    else:
        return sentence, ''

def promptify(dset_dict):
    all_outs = {}
    for author in tqdm(dset_dict):
        if author not in all_outs:
            all_outs[author] = []
        for text in dset_dict[author]:
            # try:
            first_sent, rem = split_sentence(text)

            curr_dict = {
                "prompt": f"Write an article that starts with the following: {first_sent.strip()}",
                "output": text.strip()
            }
            
            # print(curr_dict)

            curr_dict["output"] = re.sub(' +', ' ', curr_dict["output"])
            curr_dict["output"] = re.sub('\t', '', curr_dict["output"])
            curr_dict["output"] = re.sub('\r', '', curr_dict["output"])
            curr_dict["output"] = re.sub('\xa0', '', curr_dict["output"])
            curr_dict["output"] = ' '.join(curr_dict["output"].split())

            all_outs[author].append(curr_dict)
                

    return all_outs

def write_aa_dataset(data: Dict, file_path: str) -> None:
    # Save JSON data as PKL
    with open(file_path, 'wb') as f:
        pickle.dump(data, f)
        


In [3]:
dataset_procs = [
    (train_data, "train"), 
    (val_data, "val"), 
    (test_data, "test"), 
]

for dataset, name in dataset_procs:
    
    curr_dataset = promptify(dataset)

    if name == "train": 
        for k in curr_dataset:
            curr_dataset[k] = curr_dataset[k][:7]

    if name in ["val", "test"]:
        for k in curr_dataset:
            curr_dataset[k] = curr_dataset[k][:3]
            
    write_aa_dataset(curr_dataset, output_path + f"/ccat50_{name}.pkl")

  8%|▊         | 4/50 [00:00<00:01, 39.63it/s]

100%|██████████| 50/50 [00:01<00:00, 34.24it/s]


In [4]:
len(set(train_data[0]+ val_data[0] + test_data[0]))

100

In [5]:
curr_dataset[0]

[{'prompt': 'Write an article that starts with the following: U.S. Senators on Tuesday sharply criticized a new Securities and Exchange Commission rule forcing companies to disclose their use of derivatives.',
  'output': 'U.S. Senators on Tuesday sharply criticized a new Securities and Exchange Commission rule forcing companies to disclose their use of derivatives. Both the SEC and the Financial Accounting Standards Board have issued proposals to make companies disclose more about derivatives use following some high-profile losses on the complex instruments in 1994. Derivatives, financial instruments such as options and futures whose value is based on an underlying stock or commodity price, were involved in the bankruptcy of Orange County, Calif., and losses exceeding $100 million at Procter &amp; Gamble Co. The SEC adopted its rules last month, while the FASB is still working on its proposal. On Capitol Hill on Tuesday, senators took aim at both approaches, charging that the added ex

In [6]:
for name in ["train", "val", "test"]:
    with open(output_path + f"/ccat50_{name}.pkl", 'rb') as f:
        data = pickle.load(f)
        print(f"Number of authors in {name} dataset: {len(data)}")
        print(f"Number of articles per author in {name} dataset: {len(data[0])}")
        # print(f"Prompt: {data[0][0]['prompt']}")
        # print(f"Output: {data[0][0]['output']}")
        print("\n\n")

Number of authors in train dataset: 50
Number of articles per author in train dataset: 7



Number of authors in val dataset: 50
Number of articles per author in val dataset: 3



Number of authors in test dataset: 50
Number of articles per author in test dataset: 3





In [7]:
import pickle

splits = ["train", "val", "test"]
benchmark = "ccat50"
data = {} 
for split in splits: 
    with open(f"./{benchmark}/processed/{benchmark}_{split}.pkl", 'rb') as pickle_file:
        data[split] = pickle.load(pickle_file)



print("# authors: ", len(data['train']))
keys = sorted(data['train'].keys())
for idx, k in enumerate(keys):
    print(f"{idx}, {k} | train: {len(data['train'][k])} val: {len(data['val'][k])} test: {len(data['test'][k])}")

# authors:  50
0, 0 | train: 7 val: 3 test: 3
1, 1 | train: 7 val: 3 test: 3
2, 2 | train: 7 val: 3 test: 3
3, 3 | train: 7 val: 3 test: 3
4, 4 | train: 7 val: 3 test: 3
5, 5 | train: 7 val: 3 test: 3
6, 6 | train: 7 val: 3 test: 3
7, 7 | train: 7 val: 3 test: 3
8, 8 | train: 7 val: 3 test: 3
9, 9 | train: 7 val: 3 test: 3
10, 10 | train: 7 val: 3 test: 3
11, 11 | train: 7 val: 3 test: 3
12, 12 | train: 7 val: 3 test: 3
13, 13 | train: 7 val: 3 test: 3
14, 14 | train: 7 val: 3 test: 3
15, 15 | train: 7 val: 3 test: 3
16, 16 | train: 7 val: 3 test: 3
17, 17 | train: 7 val: 3 test: 3
18, 18 | train: 7 val: 3 test: 3
19, 19 | train: 7 val: 3 test: 3
20, 20 | train: 7 val: 3 test: 3
21, 21 | train: 7 val: 3 test: 3
22, 22 | train: 7 val: 3 test: 3
23, 23 | train: 7 val: 3 test: 3
24, 24 | train: 7 val: 3 test: 3
25, 25 | train: 7 val: 3 test: 3
26, 26 | train: 7 val: 3 test: 3
27, 27 | train: 7 val: 3 test: 3
28, 28 | train: 7 val: 3 test: 3
29, 29 | train: 7 val: 3 test: 3
30, 30 | train:

In [11]:
print(data['test'][0][0]['output'])

U.S. Senators on Tuesday sharply criticized a new Securities and Exchange Commission rule forcing companies to disclose their use of derivatives. Both the SEC and the Financial Accounting Standards Board have issued proposals to make companies disclose more about derivatives use following some high-profile losses on the complex instruments in 1994. Derivatives, financial instruments such as options and futures whose value is based on an underlying stock or commodity price, were involved in the bankruptcy of Orange County, Calif., and losses exceeding $100 million at Procter &amp; Gamble Co. The SEC adopted its rules last month, while the FASB is still working on its proposal. On Capitol Hill on Tuesday, senators took aim at both approaches, charging that the added expenses and complications would discourage companies from properly using derivatives to reduce risk. "One of my chief concerns when I came to the Senate in 1993 was whether we had too many unnecessary rules and regulations,"